# Asyncio 进阶模式实战

> 本 Notebook 展示了异步编程中的超时控制、并发限流（Semaphore）以及基于 Queue 的背压管理。

## 1. 超时保护 (wait_for)
演示如何防止某个协程无限期卡住事件循环。

In [ ]:
import asyncio

async def long_io():
    await asyncio.sleep(5)
    return "Finished"

async def run_with_timeout():
    try:
        # 限制 1 秒内必须完成
        await asyncio.wait_for(long_io(), timeout=1.0)
    except asyncio.TimeoutError:
        print("Operation timed out!")

await run_with_timeout()

## 2. 信号量限流 (Semaphore)
控制同一时刻允许进入临界区的协程数量。

In [ ]:
sem = asyncio.Semaphore(2)

async def task(i):
    async with sem:
        print(f"Task {i} starts")
        await asyncio.sleep(0.5)
        print(f"Task {i} ends")

await asyncio.gather(*(task(i) for i in range(5)))

## 3. 异步队列 (Queue)
实现经典的生产者-消费者解耦。

In [ ]:
q = asyncio.Queue()

async def prod():
    for i in range(3): 
        await q.put(f"msg_{i}")
    await q.put(None)

async def cons():
    while True:
        m = await q.get()
        if m is None: break
        print(f"Got {m}")

await asyncio.gather(prod(), cons())